In [ ]:
# BUY RULE:
# if (bm > 1.5 AND mom12m > 1.0) OR mve > 1.5:
#     signal = BUY

# SELL RULE:
# if bm < -1.5 OR mom12m < -1.0:
#     signal = SELL

In [1]:
import pandas as pd
import numpy as np

green_cleaned = pd.read_csv('../green cleaned.csv', dtype={'ncusip': 'string'})
green_cleaned['ret_fwd_1'] = (green_cleaned.groupby('permno')['ret_excess'].shift(-1) )

In [3]:
variables = ['datadate', 'permno', 'ticker', 'comnam', 'bm', 'mom12m', 'mve', 'ret_fwd_1']

In [17]:
import numpy as np
import pandas as pd

start_date = '2021-09-30'

# 1. Filter the data for your start date
analysis_df = green_cleaned[
    (green_cleaned['datadate'] == start_date)
][variables].copy()

# 2. Define the scoring function (REPLACE WITH THE LLM'S ACTUAL OUTPUT)
# Note: Because Pandas is vectorized, you can just pass the dataframe columns directly
def calculate_buy_probability(df):
    mve = df['mve']
    bm = df['bm']
    mom12m = df['mom12m']
    
    # --- PASTE THE LLM'S FORMULA HERE ---
    # Python native **2 works perfectly fine here in Pandas!
    score = mve * bm + bm * mom12m + mom12m * mom12m
    
    return score

# 3. Apply the scoring function to create a new 'score' column
analysis_df['score'] = calculate_buy_probability(analysis_df)

# Drop any rows where missing data caused a NaN score (to prevent ranking errors)
analysis_df = analysis_df.dropna(subset=['score'])

# 4. Rank the scores into percentiles (0.0 to 1.0)
# pct=True means the lowest score is near 0.0, and the highest score is 1.0
analysis_df['score_percentile'] = analysis_df['score'].rank(pct=True)

# 5. Define BUY (Top 10% / > 0.90) and SELL (Bottom 10% / <= 0.10)
conditions = [
    analysis_df['score_percentile'] >= 0.90,  # Top Decile
    analysis_df['score_percentile'] <= 0.10   # Bottom Decile
]

choices = ['BUY', 'SELL']

# Create the 'signal' column
analysis_df['signal'] = np.select(conditions, choices, default='HOLD')

# 6. Extract the portfolios (I added 'score' to the columns so you can verify the ranking)
output_cols = ['datadate', 'permno', 'ticker', 'comnam', 'score', 'ret_fwd_1']

buys = analysis_df[analysis_df['signal'] == 'BUY'][output_cols].sort_values(by='score', ascending=False)
sells = analysis_df[analysis_df['signal'] == 'SELL'][output_cols].sort_values(by='score', ascending=True)

print("\n--- BUY (Top 10%) ---")
with pd.option_context('display.max_rows', None):
    print(buys)

print("\n--- SELL (Bottom 10%) ---")
print(sells)

# 7. Save to CSV
buys.to_csv('buys_2021.csv', index=False)
sells.to_csv('sells_2021.csv', index=False)


--- BUY (Top 10%) ---
          datadate  permno ticker                            comnam  \
231563  2021-09-30   81055    COF        CAPITAL ONE FINANCIAL CORP   
121846  2021-09-30   38703    WFC              WELLS FARGO & CO NEW   
115885  2021-09-30   34817    NUE                        NUCOR CORP   
253909  2021-09-30   88661    TPR                      TAPESTRY INC   
33968   2021-09-30   15069    MRO                 MARATHON OIL CORP   
250620  2021-09-30   87137    DVN             DEVON ENERGY CORP NEW   
57564   2021-09-30   19583    IVZ                       INVESCO LTD   
266595  2021-09-30   92121    DFS       DISCOVER FINANCIAL SERVICES   
233810  2021-09-30   81774    FCX              FREEPORT MCMORAN INC   
96460   2021-09-30   25785      F                 FORD MOTOR CO DEL   
229464  2021-09-30   80191    ALB                    ALBEMARLE CORP   
192260  2021-09-30   64282   BBWI             BATH & BODY WORKS INC   
221147  2021-09-30   77496    MHK             MOHAWK I

In [10]:
start_date = '2021-12-31'

# 1. Filter the data for your start date
analysis_df = green_cleaned[
    (green_cleaned['datadate'] == start_date)
][variables].copy()

# 2. Define the scoring function (REPLACE WITH THE LLM'S ACTUAL OUTPUT)
# Note: Because Pandas is vectorized, you can just pass the dataframe columns directly
def calculate_buy_probability(df):
    mve = df['mve']
    bm = df['bm']
    mom12m = df['mom12m']
    
    # --- PASTE THE LLM'S FORMULA HERE ---
    # Python native **2 works perfectly fine here in Pandas!
    score = mve*mve + 2*bm*mom12m - 0.5*bm*bm
    
    return score

# 3. Apply the scoring function to create a new 'score' column
analysis_df['score'] = calculate_buy_probability(analysis_df)

# Drop any rows where missing data caused a NaN score (to prevent ranking errors)
analysis_df = analysis_df.dropna(subset=['score'])

# 4. Rank the scores into percentiles (0.0 to 1.0)
# pct=True means the lowest score is near 0.0, and the highest score is 1.0
analysis_df['score_percentile'] = analysis_df['score'].rank(pct=True)

# 5. Define BUY (Top 10% / > 0.90) and SELL (Bottom 10% / <= 0.10)
conditions = [
    analysis_df['score_percentile'] >= 0.90,  # Top Decile
    analysis_df['score_percentile'] <= 0.10   # Bottom Decile
]

choices = ['BUY', 'SELL']

# Create the 'signal' column
analysis_df['signal'] = np.select(conditions, choices, default='HOLD')

# 6. Extract the portfolios (I added 'score' to the columns so you can verify the ranking)
output_cols = ['datadate', 'permno', 'ticker', 'comnam', 'score', 'ret_fwd_1']

buys = analysis_df[analysis_df['signal'] == 'BUY'][output_cols].sort_values(by='score', ascending=False)
sells = analysis_df[analysis_df['signal'] == 'SELL'][output_cols].sort_values(by='score', ascending=True)

print("\n--- BUY (Top 10%) ---")
with pd.option_context('display.max_rows', None):
    print(buys)

print("\n--- SELL (Bottom 10%) ---")
print(sells)

# 7. Save to CSV
buys.to_csv('buys_2022.csv', index=False)
sells.to_csv('sells_2022.csv', index=False)


--- BUY (Top 10%) ---
          datadate  permno ticker                           comnam      score  \
33971   2021-12-31   15069    MRO                MARATHON OIL CORP  18.842852   
24073   2021-12-31   13641   FANG           DIAMONDBACK ENERGY INC  12.917693   
121849  2021-12-31   38703    WFC             WELLS FARGO & CO NEW  11.124061   
261428  2021-12-31   90386    MOS                    MOSAIC CO NEW   8.408912   
13274   2021-12-31   11850    XOM                 EXXON MOBIL CORP   7.884271   
179086  2021-12-31   59408    BAC             BANK OF AMERICA CORP   7.616382   
92178   2021-12-31   25081    CMA                     COMERICA INC   6.688718   
248967  2021-12-31   86868     GS          GOLDMAN SACHS GROUP INC   6.537644   
231566  2021-12-31   81055    COF       CAPITAL ONE FINANCIAL CORP   5.922838   
96463   2021-12-31   25785      F                FORD MOTOR CO DEL   5.824810   
39323   2021-12-31   16431     LW         LAMB WESTON HOLDINGS INC   5.771778   
20232

In [12]:
start_date = '2022-12-31'

# 1. Filter the data for your start date
analysis_df = green_cleaned[
    (green_cleaned['datadate'] == start_date)
][variables].copy()

# 2. Define the scoring function (REPLACE WITH THE LLM'S ACTUAL OUTPUT)
# Note: Because Pandas is vectorized, you can just pass the dataframe columns directly
def calculate_buy_probability(df):
    mve = df['mve']
    bm = df['bm']
    mom12m = df['mom12m']
    
    # --- PASTE THE LLM'S FORMULA HERE ---
    # Python native **2 works perfectly fine here in Pandas!
    score = mve + bm * bm + mom12m
    
    return score

# 3. Apply the scoring function to create a new 'score' column
analysis_df['score'] = calculate_buy_probability(analysis_df)

# Drop any rows where missing data caused a NaN score (to prevent ranking errors)
analysis_df = analysis_df.dropna(subset=['score'])

# 4. Rank the scores into percentiles (0.0 to 1.0)
# pct=True means the lowest score is near 0.0, and the highest score is 1.0
analysis_df['score_percentile'] = analysis_df['score'].rank(pct=True)

# 5. Define BUY (Top 10% / > 0.90) and SELL (Bottom 10% / <= 0.10)
conditions = [
    analysis_df['score_percentile'] >= 0.90,  # Top Decile
    analysis_df['score_percentile'] <= 0.10   # Bottom Decile
]

choices = ['BUY', 'SELL']

# Create the 'signal' column
analysis_df['signal'] = np.select(conditions, choices, default='HOLD')

# 6. Extract the portfolios (I added 'score' to the columns so you can verify the ranking)
output_cols = ['datadate', 'permno', 'ticker', 'comnam', 'score', 'ret_fwd_1']

buys = analysis_df[analysis_df['signal'] == 'BUY'][output_cols].sort_values(by='score', ascending=False)
sells = analysis_df[analysis_df['signal'] == 'SELL'][output_cols].sort_values(by='score', ascending=True)

print("\n--- BUY (Top 10%) ---")
with pd.option_context('display.max_rows', None):
    print(buys)

print("\n--- SELL (Bottom 10%) ---")
print(sells)

# 7. Save to CSV
buys.to_csv('buys_2023.csv', index=False)
sells.to_csv('sells_2023.csv', index=False)


--- BUY (Top 10%) ---
          datadate  permno ticker                            comnam  \
252691  2022-12-31   87842    MET                       METLIFE INC   
200393  2022-12-31   66800    AIG  AMERICAN INTERNATIONAL GROUP INC   
257485  2022-12-31   89258    PRU          PRUDENTIAL FINANCIAL INC   
204737  2022-12-31   70519      C                     CITIGROUP INC   
177566  2022-12-31   59248    TAP          MOLSON COORS BEVERAGE CO   
153758  2022-12-31   49015    LNC             LINCOLN NATIONAL CORP   
101093  2022-12-31   26710      L                        LOEWS CORP   
58923   2022-12-31   20057   VTRS                       VIATRIS INC   
35006   2022-12-31   15408    KHC                    KRAFT HEINZ CO   
86783   2022-12-31   24328    EQT                        E Q T CORP   
198237  2022-12-31   66093      T                       A T & T INC   
33983   2022-12-31   15069    MRO                 MARATHON OIL CORP   
37424   2022-12-31   15707    HPE     HEWLETT PACKARD 

In [15]:
start_date = '2023-12-31'

# 1. Filter the data for your start date
analysis_df = green_cleaned[
    (green_cleaned['datadate'] == start_date)
][variables].copy()

# 2. Define the scoring function (REPLACE WITH THE LLM'S ACTUAL OUTPUT)
# Note: Because Pandas is vectorized, you can just pass the dataframe columns directly
def calculate_buy_probability(df):
    mve = df['mve']
    bm = df['bm']
    mom12m = df['mom12m']
    
    # --- PASTE THE LLM'S FORMULA HERE ---
    # Python native **2 works perfectly fine here in Pandas!
    score = 0.5 * mve + 0.3 * (bm * bm) + 0.2 * (mom12m * mom12m * mom12m)
    
    return score

# 3. Apply the scoring function to create a new 'score' column
analysis_df['score'] = calculate_buy_probability(analysis_df)

# Drop any rows where missing data caused a NaN score (to prevent ranking errors)
analysis_df = analysis_df.dropna(subset=['score'])

# 4. Rank the scores into percentiles (0.0 to 1.0)
# pct=True means the lowest score is near 0.0, and the highest score is 1.0
analysis_df['score_percentile'] = analysis_df['score'].rank(pct=True)

# 5. Define BUY (Top 10% / > 0.90) and SELL (Bottom 10% / <= 0.10)
conditions = [
    analysis_df['score_percentile'] >= 0.90,  # Top Decile
    analysis_df['score_percentile'] <= 0.10   # Bottom Decile
]

choices = ['BUY', 'SELL']

# Create the 'signal' column
analysis_df['signal'] = np.select(conditions, choices, default='HOLD')

# 6. Extract the portfolios (I added 'score' to the columns so you can verify the ranking)
output_cols = ['datadate', 'permno', 'ticker', 'comnam', 'score', 'ret_fwd_1']

buys = analysis_df[analysis_df['signal'] == 'BUY'][output_cols].sort_values(by='score', ascending=False)
sells = analysis_df[analysis_df['signal'] == 'SELL'][output_cols].sort_values(by='score', ascending=True)

print("\n--- BUY (Top 10%) ---")
with pd.option_context('display.max_rows', None):
    print(buys)

print("\n--- SELL (Bottom 10%) ---")
print(sells)

# 7. Save to CSV
buys.to_csv('buys_2024.csv', index=False)
sells.to_csv('sells_2024.csv', index=False)


--- BUY (Top 10%) ---
          datadate  permno ticker                            comnam     score  \
248156  2023-12-31   86580   NVDA                       NVIDIA CORP  6.671335   
210700  2023-12-31   75510   ADBE                         ADOBE INC  6.493033   
23691   2023-12-31   13407   META                META PLATFORMS INC  6.412249   
15325   2023-12-31   12060     GE               GENERAL ELECTRIC CO  6.268347   
167037  2023-12-31   54148    PHM                   PULTE GROUP INC  6.212380   
158109  2023-12-31   50876    LLY                    LILLY ELI & CO  5.648624   
268618  2023-12-31   93002   AVGO                      BROADCOM INC  5.508996   
204749  2023-12-31   70519      C                     CITIGROUP INC  4.526090   
76005   2023-12-31   22976    WBD         WARNER BROS DISCOVERY INC  3.859311   
17568   2023-12-31   12369     GM                 GENERAL MOTORS CO  3.805151   
231590  2023-12-31   81055    COF        CAPITAL ONE FINANCIAL CORP  3.785551   
31902